<a href="https://colab.research.google.com/github/K415mm/mcp_course_workshops/blob/main/Workshop_02_Threat_Hunting/02_Threat_Hunting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️ Workshop 2: Threat Hunting with MCP

Welcome to the second workshop of the RAISEGUARD Advanced Agentic AI Course!

**Goal:** Build an MCP-powered threat hunting workflow that takes a hypothesis, generates structured SIEM queries against a log file, enriches findings, and produces a hunting report with supporting evidence.

### Setup & Libraries
Run the cell below to install our dependencies.

In [ ]:
!pip install groq langchain-groq mcp requests -q
print("✅ Libraries installed successfully!")

### API Keys
For this workshop, you will need:
1. **Groq API Key** (For our LLM Brain)
2. **AbuseIPDB API Key** (For IP enrichment during the hunt)

Add them to the Google Colab **Secrets (🔑)** tab on the left menu, then run this cell.

In [ ]:
import os
from google.colab import userdata

keys = ["GROQ_API_KEY", "ABUSEIPDB_KEY"]
for k in keys:
    try:
        os.environ[k] = userdata.get(k)
        print(f"✅ {k} successfully loaded!")
    except userdata.SecretNotFoundError:
        print(f"⚠️ WARNING: {k} missing from Colab Secrets.")
        os.environ[k] = "" 

--- 
## 💾 Lab Data Setup
First, we need to generate our synthetic Endpoint Log data for the hunt. Run this cell to create our `sample_logs.json` file.

In [ ]:
import json

sample_logs = [
  {"timestamp": "2026-03-09T21:10:00Z", "host": "WIN-DESK-04", "process": "schtasks.exe", "parent": "cmd.exe", "cmdline": "schtasks /create /tn UpdateHelper /tr C:\\Users\\Public\\upd.exe /sc onlogon", "user": "jsmith"},
  {"timestamp": "2026-03-09T21:10:05Z", "host": "WIN-DESK-04", "process": "upd.exe", "parent": "svchost.exe", "cmdline": "upd.exe --connect 185.220.101.45:443", "user": "SYSTEM"},
  {"timestamp": "2026-03-09T21:15:00Z", "host": "WIN-DESK-07", "process": "schtasks.exe", "parent": "powershell.exe", "cmdline": "schtasks /create /tn SysCheck /tr C:\\Temp\\sys.bat /sc daily", "user": "admin"},
  {"timestamp": "2026-03-09T21:20:00Z", "host": "WIN-DESK-12", "process": "wscript.exe", "parent": "excel.exe", "cmdline": "wscript C:\\Users\\Public\\load.vbs", "user": "alee"},
  {"timestamp": "2026-03-09T21:22:00Z", "host": "WIN-DESK-12", "process": "cmd.exe", "parent": "wscript.exe", "cmdline": "cmd /c powershell -enc JAB...", "user": "alee"}
]

with open("sample_logs.json", "w") as f:
    json.dump(sample_logs, f, indent=2)

print("✅ Created sample_logs.json with 5 endpoint events!")

--- 
## 🟢 Beginner Profile: Building the Log Parsing Scripts
Before we make an autonomous agent, we need to build the raw Python tools that interface with our log data. Run this cell to lay the foundation.

In [ ]:
LOG_PATH = "sample_logs.json"

def search_logs_by_process(process_name: str) -> dict:
    """Search endpoint logs for all events involving a specific process name."""
    try:
        with open(LOG_PATH) as f:
            logs = json.load(f)
        hits = [e for e in logs if process_name.lower() in e.get("process", "").lower()]
        return {"process": process_name, "hit_count": len(hits), "events": hits, "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

def get_events_for_host(hostname: str) -> dict:
    """Return all log events for a specific endpoint hostname."""
    try:
        with open(LOG_PATH) as f:
            logs = json.load(f)
        hits = [e for e in logs if e.get("host", "").lower() == hostname.lower()]
        return {"host": hostname, "event_count": len(hits), "events": hits, "status": "ok"}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

# Let's test it!
print("Testing Process Search:\n", json.dumps(search_logs_by_process("schtasks"), indent=2))

--- 
## 🟡 Intermediate Profile: Exposing Tools via FastMCP
Now we will officially expose our local hunt tools alongside our CTI Enrichment tool.

In [ ]:
from mcp.server.fastmcp import FastMCP
import requests

mcp = FastMCP("Threat Hunting Server")
ABUSE_KEY = os.environ.get("ABUSEIPDB_KEY", "")

@mcp.tool()
def hunt_schtasks() -> dict:
    """Hunts for schtasks.exe using our log searching tool."""
    return search_logs_by_process("schtasks")

@mcp.tool()
def enrich_ip(ip_address: str) -> dict:
    """Check an IPv4 address for threat intelligence using AbuseIPDB."""
    try:
        resp = requests.get(
            "https://api.abuseipdb.com/api/v2/check",
            headers={"Key": ABUSE_KEY, "Accept": "application/json"},
            params={"ipAddress": ip_address, "maxAgeInDays": 90},
            timeout=10
        ).json().get("data", {})
        return {"ip": ip_address, "abuse_confidence": resp.get("abuseConfidenceScore", 0)}
    except Exception as e:
        return {"status": "error", "reason": str(e)}

print("✅ FastMCP Server initialized with tools:", [t.name for t in mcp._tool_manager.get_tools()])

--- 
## 🔴 Advanced Profile: The Autonomous Hunting Agent
We now provide the agent with a formal Hunting Hypothesis, our logging tools, and the CTI enrichment tool. 

In [ ]:
from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

# Langchain compatible tools
@tool
def ai_search_process(process: str) -> str:
    """Search endpoint logs for all events involving a specific process name (e.g. schtasks)."""
    return str(search_logs_by_process(process))

@tool
def ai_get_host_events(host: str) -> str:
    """Return all log events for a specific endpoint hostname."""
    return str(get_events_for_host(host))

@tool
def ai_enrich_ip(ip: str) -> str:
    """Pass an IP to AbuseIPDB to get its threat score."""
    return str(enrich_ip(ip))

tools = [ai_search_process, ai_get_host_events, ai_enrich_ip]

agent = initialize_agent(
    tools, llm, agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

print("🚀 Agent Pipeline Ready!")

In [ ]:
prompt = f"""
I want to threat hunt for scheduled task persistence using LOLBins.
MITRE ATT&CK technique: T1053.005 (Scheduled Task/Job)

Please:
1. Search for schtasks.exe activity.
2. Review the logs of the hosts that executed schtasks.
3. If you find any external IP connections during your host review, enrich them using AbuseIPDB.
4. Produce a final hunting report with a VERDICT (CONFIRMED/DENIED), EVIDENCE SUMMARY, and RECOMMENDED ACTIONS.
"""

# Kick off the autonomous loop!
agent.run(prompt)